# Creating a TF-IDF baseline

## Prepare Input
We use the approved 768/512 chunking settings (chunk length / chunk overlap) to establish a TF-IDF baseline. 



In [5]:
import hashlib
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GATE3_DIR = PROJECT_ROOT / "data" / "gate3"

def load_chunks(path):
    '''Loads chunks from a CSV file and checks every chunk's text against its recorded text_hash.'''
    df = pd.read_csv(path)
    # normalize line endings on chunk files to account for Git rewriting LF -> CRLF
    # TODO: remove manual replacement once .gitattributes is in main and local copies get refreshed 
    df["text"] = df["text"].str.replace("\r\n", "\n", regex=False)
    hashes = df["text"].map(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest())
    bad = (hashes != df["text_hash"]).sum()
    assert bad == 0, f"{bad} chunks in {path.name} fail the text_hash check"
    return df

train_chunks = load_chunks(GATE3_DIR / "train_chunks_768_512.csv")
val_chunks = load_chunks(GATE3_DIR / "validation_chunks_768_512.csv")


## Normalize text

1. Unicode NFKC normalization
2. Unicode case-folding
3. Consistent equivalents for curly quotes and dashes
4. Collapse each run of whitespace to one space



In [6]:
# importing from a script, since multiple notebooks will need the same steps
from text_normalization import normalize_text

train_chunks["norm_text"] = train_chunks["text"].map(normalize_text)
val_chunks["norm_text"] = val_chunks["text"].map(normalize_text)


## TF-IDF Fixed Settings

Use one shared TF-IDF vectorizer fitted only on model-training chunk text. Validation and official-test text may use the fitted vectorizer but must not change its vocabulary or weights. When training each category's classifier, exclude the rows masked for that category.

Use these fixed vectorizer settings:

- normalized text produced with the same four steps used by the keyword baseline;
- word tokens matched by `(?u)\b\w+\b`, including one-character and numeric tokens;
- single words and two-word phrases: `ngram_range=(1, 2)`;
- `min_df` is the candidate value `2` or `5`;
- `use_idf=True`;
- `smooth_idf=True`;
- `sublinear_tf=True`;
- L2 row normalization;
- `max_df=1.0`; and
- no maximum feature limit.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

def make_vectorizer(min_df):
    '''Builds the TF-IDF vectorizer with the fixed settings; only min_df varies between configurations.'''
    return TfidfVectorizer(
        lowercase=False,               # normalize_text already case-folds
        token_pattern=r"(?u)\b\w+\b",  # keep one-character and numeric tokens
        ngram_range=(1, 2), min_df=min_df, max_df=1.0, max_features=None, stop_words=None,
        use_idf=True, smooth_idf=True, sublinear_tf=True, norm="l2",
    )

## Evaluating 6 Configurations
using every combination of:
- `min_df`: 2 or 5; and
- `C`: 0.1, 1.0, or 10.0.

### Labels and masks

The label files have one row per (chunk, category). They're reshaped into chunk × category tables with rows in the same order as the chunk files, so row `i` of a label table matches row `i` of the TF-IDF matrix. Masked rows (`mask == 1`) are excluded separately for each category during both training and threshold selection.

In [ ]:
import numpy as np

def load_label_matrices(path, chunks):
    '''Pivots long-format chunk labels into (chunk x category) target and mask tables,
    with rows in the same order as the chunk table (and so the TF-IDF matrix).'''
    
    labels = pd.read_csv(path)
    targets = labels.pivot(index="chunk_id", columns="category", values="target")
    masks = labels.pivot(index="chunk_id", columns="category", values="mask")
    assert set(targets.index) == set(chunks["chunk_id"]), f"{path.name} chunk ids don't match the chunk file"
    return targets.loc[chunks["chunk_id"]], masks.loc[chunks["chunk_id"]].astype(bool)

train_targets, train_masks = load_label_matrices(GATE3_DIR / "train_chunk_labels_768_512.csv", train_chunks)
val_targets, val_masks = load_label_matrices(GATE3_DIR / "validation_chunk_labels_768_512.csv", val_chunks)

CATEGORIES = sorted(train_targets.columns)
assert len(CATEGORIES) == 10 and CATEGORIES == sorted(val_targets.columns)

MIN_DF_VALUES = [2, 5]
C_VALUES = [0.1, 1.0, 10.0]
SEED = 20260902

### Training and threshold selection

For each configuration, one `LinearSVC` is trained per each of the 10 categories, with no resampling. A configuration fails if any of its classifiers lacks positive or negative training examples, does not converge, or produces a non-finite coefficient or margin.

Thresholds are chosen on the unmasked validation rows. Every distinct validation margin is a candidate, and a margin at or above the threshold is a detection. The threshold with the highest F1 is selected, breaking ties by higher recall and then by the lower threshold. Every candidate is kept with its confusion counts and metrics.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.svm import LinearSVC

def metrics(tp, fp, fn, tn):
    '''Precision, recall and F1, returning 0 when a denominator is 0.'''
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return precision, recall, f1

def threshold_search(margins, y):
    '''Treats every distinct validation margin as a threshold (margin >= threshold is a detection)
    and returns the confusion counts and metrics for each, highest threshold first.'''
    order = np.argsort(-margins, kind="stable")
    m, yy = margins[order], y[order]
    tp_cum, fp_cum = np.cumsum(yy), np.cumsum(1 - yy)

    # last position of each distinct margin = everything at or above that threshold
    last = np.r_[np.flatnonzero(m[1:] != m[:-1]), len(m) - 1]
    tp, fp = tp_cum[last], fp_cum[last]
    fn, tn = yy.sum() - tp, (1 - yy).sum() - fp
    rows = [(t, a, b, c, d, a + b, *metrics(a, b, c, d)) for t, a, b, c, d in zip(m[last], tp, fp, fn, tn)]
    return pd.DataFrame(rows, columns=["threshold", "tp", "fp", "fn", "tn", "predicted_positive",
                                       "precision", "recall", "f1"])

def select_threshold(search):
    '''Highest F1, then higher recall, then lower threshold.'''
    return search.sort_values(["f1", "recall", "threshold"], ascending=[False, False, True]).iloc[0]

def train_category(X_train, y_train, X_val, y_val, C):
    '''Trains one category's LinearSVC and scores the validation rows. Callers pass only that
    category's unmasked rows. Returns (classifier, validation margins, failure reason or None).'''

    if len(np.unique(y_train)) < 2:
        return None, None, "training rows lack a positive or negative example"
    
    if len(np.unique(y_val)) < 2:
        return None, None, "validation rows lack a positive or negative example"

    clf = LinearSVC(penalty="l2", loss="squared_hinge", dual=True, fit_intercept=True, tol=1e-4,
                    max_iter=10000, class_weight="balanced", random_state=SEED, C=C)
    
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        clf.fit(X_train, y_train)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught) or clf.n_iter_ >= clf.max_iter:
        return clf, None, f"did not converge in {clf.max_iter} iterations"
    if not (np.isfinite(clf.coef_).all() and np.isfinite(clf.intercept_).all()):
        return clf, None, "non-finite coefficient"

    margins = clf.decision_function(X_val)
    if not np.isfinite(margins).all():
        return clf, margins, "non-finite validation margin"
    return clf, margins, None

In [10]:
vectorizers = {}    # min_df -> TfidfVectorizer fitted on training chunks
classifiers = {}    # (config, category) -> fitted LinearSVC
results = []        # one row per (config, category): selected threshold and its metrics
searches = []       # every candidate threshold for every (config, category)
val_margins = []    # every unmasked validation margin
failures = []       # one row per failed (config, category)

for min_df in MIN_DF_VALUES:
    vectorizers[min_df] = make_vectorizer(min_df)
    X_train = vectorizers[min_df].fit_transform(train_chunks["norm_text"])  # vocabulary from training chunks only
    X_val = vectorizers[min_df].transform(val_chunks["norm_text"])
    print(f"min_df={min_df}: {X_train.shape[1]:,} features")

    for C in C_VALUES:
        config = f"min_df={min_df}, C={C}"
        for category in CATEGORIES:
            # drop this category's masked rows; other categories keep them
            train_rows = ~train_masks[category].to_numpy()
            val_rows = ~val_masks[category].to_numpy()
            y_train = train_targets[category].to_numpy()[train_rows]
            y_val = val_targets[category].to_numpy()[val_rows]

            clf, margins, failure = train_category(X_train[train_rows], y_train, X_val[val_rows], y_val, C)
            if failure:
                failures.append({"config": config, "min_df": min_df, "C": C,
                                 "category": category, "reason": failure})
                continue
            classifiers[config, category] = clf

            val_ids = val_chunks["chunk_id"].to_numpy()[val_rows]
            val_margins.append(pd.DataFrame({"config": config, "category": category, "chunk_id": val_ids,
                                             "target": y_val, "margin": margins}))
            search = threshold_search(margins, y_val)
            searches.append(search.assign(config=config, category=category))
            best = select_threshold(search)
            results.append({"config": config, "min_df": min_df, "C": C, "category": category,
                            "n_iter": clf.n_iter_, "support": int(y_val.sum()), "n_val_rows": len(y_val),
                            **best.to_dict(),
                            "all_positive": best["predicted_positive"] == len(y_val)})
        print(f"  {config}: done")

results = pd.DataFrame(results)
# best.to_dict() turns counts into floats; restore them
results = results.astype({k: int for k in ["tp", "fp", "fn", "tn", "predicted_positive"]})
searches = pd.concat(searches, ignore_index=True)
val_margins = pd.concat(val_margins, ignore_index=True)
failures = pd.DataFrame(failures, columns=["config", "min_df", "C", "category", "reason"])

min_df=2: 429,844 features
  min_df=2, C=0.1: done
  min_df=2, C=1.0: done
  min_df=2, C=10.0: done
min_df=5: 160,988 features
  min_df=5, C=0.1: done
  min_df=5, C=1.0: done
  min_df=5, C=10.0: done


## Selecting a configuration

Configurations with any failed classifier are excluded (see `failures`). Among the rest, macro-F1 is the plain average of the 10 category F1 scores, and micro metrics pool `TP`/`FP`/`FN`/`TN` across categories. The configuration is chosen by:

1. highest macro-F1;
2. then `min_df=5` if there is a tie; and
3. then the lower `C` value if a tie remains.

A category whose selected threshold flags every validation row is listed under `weak_all_positive`.

In [11]:
failed_configs = set(failures["config"])
valid = results[~results["config"].isin(failed_configs)]

def summarize(group):
    tp, fp, fn, tn = (group[k].sum() for k in ["tp", "fp", "fn", "tn"])
    micro_p, micro_r, micro_f1 = metrics(tp, fp, fn, tn)
    return pd.Series({"macro_f1": group["f1"].mean(), "micro_precision": micro_p,
                      "micro_recall": micro_r, "micro_f1": micro_f1,
                      "weak_all_positive": group.loc[group["all_positive"], "category"].tolist()})

config_summary = (valid.groupby(["config", "min_df", "C"]).apply(summarize, include_groups=False)
                  .reset_index()
                  # highest macro-F1, then min_df=5, then lower C
                  .sort_values(["macro_f1", "min_df", "C"], ascending=[False, False, True])
                  .reset_index(drop=True))
assert (valid.groupby("config")["category"].nunique() == 10).all()

selected = config_summary.iloc[0]
selected_config = selected["config"]
print(f"Selected configuration: {selected_config}")
if failed_configs:
    print(f"Failed configurations: {sorted(failed_configs)}")

# fitted pieces of the selected configuration, for scoring the official test set later
selected_vectorizer = vectorizers[selected["min_df"]]
selected_classifiers = {category: classifiers[selected_config, category] for category in CATEGORIES}
selected_thresholds = valid[valid["config"] == selected_config].set_index("category")["threshold"].to_dict()

config_summary

Selected configuration: min_df=2, C=10.0


,config,min_df,C,macro_f1,micro_precision,micro_recall,micro_f1,weak_all_positive
0,"min_df=2, C=10.0",2,10.0,0.711664,0.762162,0.735469,0.748578,[]
1,"min_df=5, C=10.0",5,10.0,0.710839,0.739098,0.745156,0.742115,[]
2,"min_df=5, C=1.0",5,1.0,0.705872,0.741862,0.730253,0.736012,[]
3,"min_df=2, C=1.0",2,1.0,0.705813,0.769541,0.711624,0.739450,[]
4,"min_df=5, C=0.1",5,0.1,0.676890,0.730558,0.692996,0.711281,[]
5,"min_df=2, C=0.1",2,0.1,0.669805,0.712750,0.691505,0.701967,[]


In [12]:
# per-category results for the selected configuration
(valid[valid["config"] == selected_config]
 [["category", "support", "threshold", "tp", "fp", "fn", "tn", "precision", "recall", "f1", "all_positive"]]
 .sort_values("f1", ascending=False).reset_index(drop=True))

,category,support,threshold,tp,fp,fn,tn,precision,recall,f1,all_positive
0,Anti-Assignment,217,-0.238880,182,30,35,2201,0.858491,0.838710,0.848485,False
1,License Grant,207,-0.198523,165,30,42,2221,0.846154,0.797101,0.820896,False
2,Expiration Date,192,-0.114155,147,25,45,2248,0.854651,0.765625,0.807692,False
3,Cap On Liability,229,-0.330011,168,25,61,2192,0.870466,0.733624,0.796209,False
4,Renewal Term,78,-0.016273,56,7,22,2388,0.888889,0.717949,0.794326,False
5,Non-Transferable License,89,-0.244827,76,28,13,2359,0.730769,0.853933,0.787565,False
6,Uncapped Liability,66,-0.310740,52,33,14,2374,0.611765,0.787879,0.688742,False
7,Exclusivity,119,-0.436379,64,53,55,2284,0.547009,0.537815,0.542373,False
8,Change Of Control,73,-0.325704,38,35,35,2364,0.520548,0.520548,0.520548,False
9,Non-Compete,72,-0.572467,39,42,33,2365,0.481481,0.541667,0.509804,False


## Saving diagnostics

The milestone asks for the threshold searches, margins and metrics of every valid configuration, plus failure details for any excluded one. These are written to `reports/tf_idf_baseline/`.

In [13]:
OUTPUT_DIR = PROJECT_ROOT / "reports" / "tf_idf_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config_summary.to_csv(OUTPUT_DIR / "config_summary.csv", index=False)
results.to_csv(OUTPUT_DIR / "category_results.csv", index=False)
searches.to_csv(OUTPUT_DIR / "threshold_searches.csv", index=False)
val_margins.to_csv(OUTPUT_DIR / "validation_margins.csv", index=False)
failures.to_csv(OUTPUT_DIR / "failures.csv", index=False)